In [1]:
%load_ext autoreload
%autoreload 2

#set the path to the root of the project dynamically
import sys
import os
os.chdir(os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir)))

In [12]:
from src.gem.engine.ecosystem_engine import EcosystemEngine
from src.gem.engine.environment_state import EnvironmentState
from src.gem.engine.ecosystem_grid_state import EcosystemGridState
from src.gem.engine.species_registry import SpeciesRegistry

from src.gem.engine.processes import apply_vegetation_growth, apply_atn_step, apply_dispersal
import numpy as np
import rasterio

processed_dir = './data/processed/'
temp_path = os.path.join(processed_dir, 'Tavg_NorthAmerica_EqualArea_100km_month_01.tif') # January example

with rasterio.open(temp_path) as src:
    temp_data = src.read(1)
    
    

world_shape = (75, 120)  # (12000 km x 7500 km with 100 km cell size)
env = EnvironmentState(world_shape)
env.add_layer("temperature", temp_data)  # Add temperature layer from raster
env.add_layer("boundary_number", np.zeros(world_shape))  # Add boundary layer (all zeros for now)
# Create 3 species: 1 plant, 2 animals
registry = SpeciesRegistry(num_species=20)
registry.add_species_to_group(
    group_name="deciduous_trees", 
    species_indices=list(range(0, 5)), 
    params={
        "base_growth_rate": 0.15, 
        "mortality_rate": 0.05,
        "max_dispersal_rate": 0.01
    }
)

registry.add_species_to_group(
    group_name="coniferous_trees", 
    species_indices=list(range(10, 15)), 
    params={
        "base_growth_rate": 0.08, 
        "mortality_rate": 0.02,
        "max_dispersal_rate": 0.02
    }
)

registry.add_species_to_group(
    group_name="herbs", 
    species_indices=list(range(5, 10)), 
    params={
        "base_growth_rate": 0.2, 
        "mortality_rate": 0.1,
        "max_dispersal_rate": 0.05
    }
)
registry.add_species_to_group(
    group_name="herbivores", 
    species_indices=list(range(15, 18)), 
    params={
        "base_growth_rate": 0.1, 
        "mortality_rate": 0.05,
        "max_dispersal_rate": 0.1
    }
)
registry.add_species_to_group(
    group_name="carnivores", 
    species_indices=list(range(18, 20)), 
    params={
        "base_growth_rate": 0.05, 
        "mortality_rate": 0.02,
        "max_dispersal_rate": 0.2
    }
)

#bigger groups for more interesting dynamics
registry.add_species_to_group(group_name="vertebrates", species_indices=list(range(15, 20))) # All vertebrates for process targeting
registry.add_species_to_group(group_name="plants", species_indices=list(range(0, 15))) # All plants for process targeting
#add feeding links (plants -> herbivores -> carnivores)
registry.add_feeding_link("plants", "herbivores")
registry.add_feeding_link("herbivores", "carnivores")


#Grid state with 20 species (15 plants, 5 animals)
grid = EcosystemGridState(world_shape, registry)

#ADDING LAYERS
#TODO make the naming uniform
grid.add_layer("metabolism_rate")  # Base metabolism rate for all species
grid.add_layer("net_growth_rate")  # Initialize net growth rate layer

#ADDING DELTA LAYERS
grid.add_layer("dispersal_delta")  # Initialize dispersal delta layer

print(f"Environment initialized with layers: {list(env.layers.keys())}")
print(f"Grid shape: {env.shape}")
# SEED INITIAL LAYERS
grid.layers["biomass"][:, :, registry.get_group_indices("plants")] = 50000.0  # Plants
grid.layers["biomass"][:, :, registry.get_group_indices("animals")] = 5000.0  # Animals

# --- 2. Build the Engine ---
model = EcosystemEngine(grid, env)

# The order here defines the pipeline sequence

model.add_process(apply_dispersal)

# --- 3. Run the Simulation ---
num_steps = 10
vertebrate_biomass_history = []
plant_biomass_history = []
for tick in range(num_steps):
    model.step()
    
    # Track Totals using the new matrix views
    # np.sum() cleanly aggregates the whole grid
    total_plant = np.sum(grid.get_layer_view("biomass", "plants"))
    total_vertebrates = np.sum(grid.get_layer_view("biomass", "vertebrates"))
    
    plant_biomass_history.append(total_plant)
    vertebrate_biomass_history.append(total_vertebrates)
    


Environment initialized with layers: ['temperature', 'boundary_number']
Grid shape: (75, 120)
